# 03 — Evaluation```bashpython -m src.metrics ../checkpoints/fastgan/final.pt --json ../docs/results.json```**KID is the headline metric, not FID.** FID's covariance estimate is badlybiased below ~2,048 samples and this dataset has ~2.2k images in total. FID isreported with its sample count attached and read as indicative only.

In [ ]:
import sys; sys.path.insert(0, "..")from pathlib import Pathimport torchimport matplotlib.pyplot as pltfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator

In [ ]:
from src.config import get_devicefrom src.data.dataset import PokemonArtworkfrom src.sample import generate, interpolate, latents, load_generator, make_grid, to_pilCKPT = Path("../checkpoints/fastgan/final.pt")device = get_device()netG = load_generator(CKPT, device)      # EMA weights, deterministic noiseds = PokemonArtwork(resolution=256, mirror=False)print("loaded", CKPT, "| dataset:", len(ds))

## Samples

In [ ]:
z = latents(36, netG.z_dim, seed=0, device=device)fig, ax = plt.subplots(figsize=(11, 11))ax.imshow(make_grid(to_pil(generate(netG, z, truncation=0.8)))); ax.axis("off")plt.show()

## Truncation sweeppsi trades diversity for fidelity: low psi pulls latents toward the mean andgives safe, samey outputs; high psi gives wild and often broken ones.

In [ ]:
psis = [0.3, 0.5, 0.7, 0.9, 1.1]z = latents(4, netG.z_dim, seed=3, device=device)fig, axes = plt.subplots(1, len(psis), figsize=(4 * len(psis), 4.6))for ax, psi in zip(axes, psis):    ax.imshow(make_grid(to_pil(generate(netG, z, truncation=psi)), cols=2)); ax.axis("off")    ax.set_title(f"psi = {psi}")plt.tight_layout(); plt.show()

## Latent interpolation (slerp)

In [ ]:
frames = interpolate(netG, seed_a=11, seed_b=77, steps=8, truncation=0.8)fig, ax = plt.subplots(figsize=(16, 2.5))ax.imshow(make_grid(frames, cols=len(frames))); ax.axis("off"); plt.show()

## Memorisation check — the result that matters mostWith only ~1,308 distinct training shapes, the first question any reviewershould ask is whether the model is reproducing training art. Top row isgenerated; bottom row is each sample's nearest training image in VGG featurespace. If they match closely, this is an expensive lookup table, not agenerative model.

In [ ]:
from src.metrics import nearest_neighbourspanel = nearest_neighbours(netG, ds, device, count=8)Path("../docs/assets").mkdir(parents=True, exist_ok=True)panel.save("../docs/assets/nearest_neighbours.png")fig, ax = plt.subplots(figsize=(16, 4.6))ax.imshow(panel); ax.axis("off")ax.set_title("top: generated    bottom: nearest training image")plt.show()

## KID and FID

In [ ]:
from src.metrics import (    InceptionFeatures,    features_from_dataset,    features_from_generator,    frechet_distance,    kernel_distance,)inception = InceptionFeatures().to(device)real = features_from_dataset(inception, ds, 16, device)fake = features_from_generator(inception, netG, 2000, 16, device, truncation=1.0)kid_mean, kid_std = kernel_distance(real, fake, subset_size=min(1000, len(real)))print(f"KID  {kid_mean:.5f} +/- {kid_std:.5f}   <- headline metric")print(f"FID  {frechet_distance(real, fake):.2f}   (n_real={len(real)}, indicative only)")

## Failure modesShip these. They are more informative than the cherry-picks, and a results pagewithout them is not an honest one.

In [ ]:
z = latents(16, netG.z_dim, seed=999, device=device)fig, ax = plt.subplots(figsize=(11, 11))ax.imshow(make_grid(to_pil(generate(netG, z, truncation=1.2)))); ax.axis("off")ax.set_title("psi = 1.2 - where it breaks")plt.show()